In [2]:
from osgeo import gdal
from osgeo import ogr
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Polygon
from shapely.ops import unary_union

In [62]:
gdf = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/idk big.geojson")

In [127]:
brt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter = ";")

In [141]:
gdf["function"].unique()

array(['rijbaan lokale weg', 'voetpad', 'parkeervlak', 'inrit',
       'rijbaan regionale weg', 'fietspad', 'ruiterpad',
       'voetgangersgebied', 'OV-baan', 'voetpad op trap',
       'rijbaan autoweg', 'transitie', 'overweg', 'spoorbaan',
       'rijbaan autosnelweg', 'woonerf', 'baan voor vliegverkeer'],
      dtype=object)

In [126]:
brt.crs

<Projected CRS: EPSG:28992>
Name: Amersfoort / RD New
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Netherlands - onshore, including Waddenzee, Dutch Wadden Islands and 12-mile offshore coastal zone.
- bounds: (3.2, 50.75, 7.22, 53.7)
Coordinate Operation:
- name: RD New
- method: Oblique Stereographic
Datum: Amersfoort
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [128]:
s = gpd.GeoSeries.from_wkt(brt["WKT_LNG_LAT"])
brt = gpd.GeoDataFrame(brt, geometry = s, crs = "EPSG:4326")

In [94]:
df = gdf[["gml_id", "function", "surfaceMaterial", "geometry"]]
df = df[df["function"].isin(["voetpad", "voetgangersgebied", "voetpad op trap"])]

In [98]:
df = df.drop_duplicates(subset = "geometry")

In [108]:
# Keep only valid geometries
pedestrian_gdf = df[df.is_valid]

# Reset index after removal
pedestrian_gdf = pedestrian_gdf.reset_index(drop=True)

# Count remaining invalid geometries
invalid_count = pedestrian_gdf[~pedestrian_gdf.is_valid].shape[0]
invalid_count


0

In [109]:
merged_pedestrian_polygon = unary_union(pedestrian_gdf.geometry)

In [111]:
merged_pedestrian_gdf = gpd.GeoDataFrame(geometry=[merged_pedestrian_polygon], crs=pedestrian_gdf.crs)

In [114]:
merged_pedestrian_gdf.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/test_big_poly_ped.csv")

In [131]:
brt = brt.to_crs(merged_pedestrian_gdf.crs)

In [132]:
clipped_pedestrian_gdf = gpd.overlay(merged_pedestrian_gdf, brt, how="intersection")


In [134]:
brt

,OBJECTNUMMER,CBS_Buurtcode,Buurtcode,Buurt,Wijkcode,Wijk,Gebiedcode,Gebied,Stadsdeelcode,Stadsdeel,Oppervlakte_m2,WKT_LNG_LAT,WKT_LAT_LNG,LNG,LAT,geometry
0,1,BU0363AC02,AC02,Leliegracht e.o.,AC,Grachtengordel-West,GA01,Centrum-West,A,Centrum,172784,"POLYGON((4.883069 52.373992,4.883385 52.374065...","POLYGON((52.373992 4.883069,52.374065 4.883385...",4.887192,52.375675,"POLYGON ((120669.396 487465.81, 120690.972 487..."
1,2,BU0363AC03,AC03,Felix Meritisbuurt,AC,Grachtengordel-West,GA01,Centrum-West,A,Centrum,202396,"POLYGON((4.882614 52.368932,4.884411 52.368862...","POLYGON((52.368932 4.882614,52.368862 4.884411...",4.885734,52.371411,"POLYGON ((120634.492 486903.035, 120756.823 48..."
2,3,BU0363AC04,AC04,Leidsegracht-Noord,AC,Grachtengordel-West,GA01,Centrum-West,A,Centrum,68876,"POLYGON((4.882603 52.368756,4.882424 52.366744...","POLYGON((52.368756 4.882603,52.366744 4.882424...",4.885242,52.367554,"POLYGON ((120633.607 486883.458, 120619.857 48..."
3,4,BU0363AD01,AD01,Stationsplein e.o.,AD,Burgwallen-Nieuwe Zijde,GA01,Centrum-West,A,Centrum,259429,"POLYGON((4.895468 52.379856,4.897528 52.378761...","POLYGON((52.379856 4.895468,52.378761 4.897528...",4.900939,52.379780,"POLYGON ((121518.161 488112.455, 121657.6 4879..."
4,5,BU0363AD02,AD02,Hemelrijk,AD,Burgwallen-Nieuwe Zijde,GA01,Centrum-West,A,Centrum,62725,"POLYGON((4.892082 52.377032,4.8942 52.376609,4...","POLYGON((52.377032 4.892082,52.376609 4.8942,5...",4.894897,52.378233,"POLYGON ((121285.468 487799.818, 121429.368 48..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513,514,BU0363AB08,AB08,Groenmarktkadebuurt,AB,Jordaan,GA01,Centrum-West,A,Centrum,49285,"POLYGON((4.874503 52.371889,4.875045 52.370683...","POLYGON((52.371889 4.874503,52.370683 4.875045...",4.876399,52.370409,"POLYGON ((120084.42 487235.916, 120120.382 487..."
514,515,BU0363AB09,AB09,Marnixbuurt-Zuid,AB,Jordaan,GA01,Centrum-West,A,Centrum,51032,"POLYGON((4.876809 52.368503,4.877088 52.368108...","POLYGON((52.368503 4.876809,52.368108 4.877088...",4.878861,52.366889,"POLYGON ((120238.807 486858.071, 120257.499 48..."
515,516,BU0363AB10,AB10,Elandsgrachtbuurt,AB,Jordaan,GA01,Centrum-West,A,Centrum,207114,"POLYGON((4.876314 52.372314,4.876179 52.371903...","POLYGON((52.372314 4.876314,52.371903 4.876179...",4.879624,52.370869,"POLYGON ((120208.084 487282.332, 120198.568 48..."
516,517,BU0363AB11,AB11,Passeerdersgrachtbuurt,AB,Jordaan,GA01,Centrum-West,A,Centrum,54476,"POLYGON((4.879075 52.367745,4.879889 52.36661,...","POLYGON((52.367745 4.879075,52.36661 4.879889,...",4.880936,52.367130,"POLYGON ((120392.543 486772.65, 120447.098 486..."


In [148]:
clipped_pedestrian_gdf["area"] = clipped_pedestrian_gdf.geometry.area
clipped_pedestrian_gdf["area_buurt"] = clipped_pedestrian_gdf.area
clipped_pedestrian_gdf

AttributeError: 'Series' object has no attribute 'area'

In [145]:
clipped_pedestrian_gdf.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/pedestrian_space_area.gpkg", driver="GPKG")